<a href="https://colab.research.google.com/github/ZIZOUTOUMOU/LAB02-AdvDB/blob/main/TP01_web_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (educational scraping project TP01)"}
DELAY = 0.5
SHOW_HTML = False

session = requests.Session()
session.headers.update(HEADERS)
print("Ready")

Ready


##  Fetch all pages and display the HTML source

In [3]:
def fetch(url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=10)
            resp.raise_for_status()
            resp.encoding = "utf-8"
            return resp.text
        except requests.RequestException as err:
            print(f"[warn] {url} (attempt {attempt}/{retries}): {err}")
            time.sleep(1)
    return None


def crawl(start_url, next_selector):
    """Follow the 'Next' link until the last page; print the HTML of each page."""
    url = start_url
    while url:
        html = fetch(url)
        if html is None:
            break
        print("=" * 80)
        print(f"PAGE: {url}  ({len(html)} characters)")
        print("=" * 80)
        print(html if SHOW_HTML else html[:500] + "\n...[truncated, set SHOW_HTML = True]...")
        yield url, html
        soup = BeautifulSoup(html, "html.parser")
        nxt = soup.select_one(next_selector)
        url = urljoin(url, nxt["href"]) if nxt else None
        time.sleep(DELAY)

##  Extract all quotes with BeautifulSoup

In [4]:
def parse_quotes(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for q in soup.select("div.quote"):
        rows.append({
            "quote": q.select_one("span.text").get_text(strip=True),
            "author": q.select_one("small.author").get_text(strip=True),
            "tags": ", ".join(t.get_text(strip=True) for t in q.select("a.tag")),
            "source_page": page_url,
        })
    return rows


quote_rows = []
for url, html in crawl("https://quotes.toscrape.com/", "li.next a"):
    quote_rows.extend(parse_quotes(html, url))

quotes_df = pd.DataFrame(quote_rows)
print(f"\nTotal quotes: {len(quotes_df)}")
quotes_df.head(10)

PAGE: https://quotes.toscrape.com/  (11021 characters)
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div cla
...[truncated, set SHOW_HTML = True]...
PAGE: https://quotes.toscrape.com/page/2/  (13699 characters)
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
       

,quote,author,tags,source_page
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",https://quotes.toscrape.com/
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",https://quotes.toscrape.com/
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",https://quotes.toscrape.com/
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",https://quotes.toscrape.com/
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",https://quotes.toscrape.com/
5,“Try not to become a man of success. Rather be...,Albert Einstein,"adulthood, success, value",https://quotes.toscrape.com/
6,“It is better to be hated for what you are tha...,André Gide,"life, love",https://quotes.toscrape.com/
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"edison, failure, inspirational, paraphrased",https://quotes.toscrape.com/
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,misattributed-eleanor-roosevelt,https://quotes.toscrape.com/
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"humor, obvious, simile",https://quotes.toscrape.com/


## Save the results to CSV with pandas

In [5]:
quotes_df.to_csv("quotes.csv", index=False, encoding="utf-8-sig")
print("quotes.csv saved:", len(quotes_df), "rows")

quotes.csv saved: 100 rows


In [6]:
def parse_books(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for b in soup.select("article.product_pod"):
        link = b.select_one("h3 a")
        rows.append({
            "title": link["title"],
            "price": b.select_one("p.price_color").get_text(strip=True),
            "rating": b.select_one("p.star-rating")["class"][1],
            "availability": b.select_one("p.availability").get_text(strip=True),
            "url": urljoin(page_url, link["href"]),
        })
    return rows


book_rows = []
for url, html in crawl("https://books.toscrape.com/", "li.next a"):
    book_rows.extend(parse_books(html, url))

books_df = pd.DataFrame(book_rows)
books_df.to_csv("books.csv", index=False, encoding="utf-8-sig")

print(f"\nbooks.csv saved: {len(books_df)} rows")
assert len(books_df) >= 1000, f"Only {len(books_df)} rows"
print("OK: at least 1,000 rows")
books_df.head(10)

PAGE: https://books.toscrape.com/  (51274 characters)
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /
...[truncated, set SHOW_HTML = True]...
PAGE: https://books.toscrape.com/catalogue/page-2.html  (50853 characters)


<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!-

,title,price,rating,availability,url
0,A Light in the Attic,£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,£22.65,One,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,Four,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,Three,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,Four,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,£52.15,One,In stock,https://books.toscrape.com/catalogue/the-black...


## Download the CSV files to your computer

In [7]:
from google.colab import files
files.download("quotes.csv")
files.download("books.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>